In [ ]:
!pip install nibabel

In [ ]:
import os, json, glob, random
import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from tqdm import tqdm

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import backend as K
from tensorflow.keras.layers import (Input, Conv3D, MaxPooling3D, UpSampling3D,
Concatenate, Activation, BatchNormalization)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam


tf.compat.v1.logging.set_verbosity(tf.compat.v1.logging.ERROR)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"


## Exploring the Dataset

In [ ]:
# Getting the Dataset path ready!

DATA_DIR = '/kaggle/input/datasets/awsaf49/brats20-dataset-training-validation/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData/'

# lists of directories with studies
train_and_val_directories = [f.path for f in os.scandir(DATA_DIR) if f.is_dir()]

# file BraTS20_Training_355 has ill formatted name for for seg.nii file
train_and_val_directories.remove(DATA_DIR +'BraTS20_Training_355')

#confirming that the dataset is accessible

cases = sorted(train_and_val_directories)
print(f"Total training cases found: {len(cases)}")
print("First 3 cases: ")
for c in cases[:3]:
    print(" ", os.path.basename(c))


test_image_seg = nib.load(case_folder + '/BraTS20_Training_001_seg.nii').get_fdata()
test_image_seg = (test_image_seg - test_image_seg.min()) / (test_image_seg.max() - test_image_seg.min())
print(test_image_seg.max())

In [ ]:
#loading and inspectin one case
def load_case(case_folder):
    # load mri sequences and label for one patient

    sequences = ["flair", "t1", "t1ce", "t2"]
    vols = []
    folder_name = os.path.basename(case_folder)

    for seq in sequences:
        path = os.path.join(case_folder, f"{folder_name}_{seq}.nii")
        vol = np.array(nib.load(path).get_fdata(), dtype = np.float32)
        vols.append(vol)

    image = np.stack(vols, axis = -1) #shape L (240, 240 ,155, 4)

    label_path = os.path.join(case_folder, f"{folder_name}_seg.nii")
    label = np.array(nib.load(label_path).get_fdata(), dtype = np.float32)

    return image, label

#load the very first case

sample_image, sample_label = load_case(cases[0])

print("Image shape :", sample_image.shape) # (240, 240, 155, 4)
print("Label shape :", sample_label.shape) # (240, 240, 155)
print("Image dtype :", sample_image.dtype)
print("Unique label values:", np.unique(sample_label).astype(int))
print()

print("Sequence value ranges: ")
for i, seq in enumerate(["FLAIR", "T1", "T1Gd", "T2"]):
    mn, mx = sample_image[..., i].min(), sample_image[..., i].max()
    print(f"  {seq}: min={mn:1f}, max = {mx:.1f}")

In [ ]:
#slice visualization

def visualize_slice(image, label, z_slice = 77, case_name = ""):
    #plotting 4 mri sequences and label for a single axial slice

    fig, axes = plt.subplots(1, 5, figsize = (20, 4))
    sequences = ["FLAIR", "T1", "T1Gd", "T2"]
    cmaps = ["gray", "gray", "gray", "gray"]

    for i, (seq, cmap) in enumerate(zip(sequences, cmaps)):
        axes[i].imshow(image[:, :, z_slice, i], cmap = cmap)
        axes[i].set_title(seq, fontsize = 13)
        axes[i].axis("off")

    # Color coded label overlay

    label_slice = label[:,:, z_slice]
    color_label = np.zeros((*label_slice.shape, 3))
    color_label[label_slice == 1] = [1, 0.3, 0.3] #red for edema
    color_label[label_slice == 2] = [0.3, 1, 0.3] #green for non-enhancing
    color_label[label_slice == 4] = [0.3, 0.5, 1] #blue for enhancing

    axes[4].imshow(image[:, :, z_slice, 0], cmap="gray")
    axes[4].imshow(color_label, alpha = 0.5)
    axes[4].set_title("Segmentation", fontsize = 13)
    axes[4].axis("off")

    # Legend
    patches = [mpatches.Patch(color=[1,0.3,0.3], label="Edema"),
                mpatches.Patch(color=[0.3,1,0.3], label="Non-Enhancing"),
                mpatches.Patch(color=[0.3,0.5,1], label="Enhancing")]
    
    axes[4].legend(handles=patches, loc="lower right", fontsize=8)
    
    title = case_name if case_name else f"Axial slice Z={z_slice}"
    fig.suptitle(title, fontsize=15, fontweight="bold")
    plt.tight_layout()
    plt.show()

#visualizing 3 different depth slices
name = os.path.basename(cases[0])
for z in [50, 77, 100]:
    visualize_slice(sample_image, sample_label, z_slice = z, case_name = f"{name}  | Z = {z}")

In [ ]:
#counting the voxels accross all the cases

#scans all the cases and calculated the label frequencies

label_counts = {
    0: 0,
    1: 0, 
    2: 0,
    4: 0
}

for case_path in tqdm(cases, desc = "Scanning Dataset"):
    folder_name = os.path.basename(case_path)
    seg_path = os.path.join(case_path, f"{folder_name}_seg.nii")
    seg = np.array(nib.load(seg_path).get_fdata(), dtype = np.uint8)
    unique, counts = np.unique(seg, return_counts = True)
    for u, c in zip(unique, counts):
        label_counts[int(u)] = label_counts.get(int(u), 0) + c

total = sum(label_counts.values())
print("\nVoxel class distribution across all 369 cases: ")
names = {
    0: "Background",
    1: "Edemea",
    2: "Non-Enhancing",
    4: "Enhancing"
}

for k, v in sorted(label_counts.items()):
    print(f" {names[k]: >15s}: {v>12,} ({100*v/total:.2f}%)")


In [ ]:
#Bar chart for showing class distribution

fig, ax = plt.subplots(figsize = (8, 4))
labels_display = [names[k] for k in [1, 2, 4]]
counts_display = [label_counts[k] for k in [1, 2, 4]]
colors = [[1, 0.3, 0.3],[0.3, 1, 0.3], [0.3, 0.5, 1]]
ax.bar(labels_display, counts_display, color = colors)
ax.set_title("Tumor voxel counts by class (background excluded)")
ax.set_ylabel("Voxel Count")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _:f"{x/1e6:.1f}M"))
plt.tight_layout()
plt.show()

## Preprocessing the Data

In [ ]:
# train/validation split

random.seed(42)

all_cases = sorted(train_and_val_directories)
random.shuffle(all_cases)

split_idx = int(0.8 * len(all_cases))
train_cases = all_cases[:split_idx]
val_cases = all_cases[split_idx:]

print(f"Training cases: {len(train_cases)}")
print(f"Validation cases: {len(val_cases)}")

In [ ]:
#extracting a random sub_volume that contains enough tumor

def get_sub_volume(image, label,
                   output_x = 128, output_y = 128,output_z = 16,
                   background_threshold = 0.95, max_tries = 1000):
    orig_x, orig_y, orig_z = 240, 240 ,155

    # remapping label 4 to 3 (enhancing tumor uses 4, so making it 3)
    label = np.copy(label)
    label[label == 4] = 3

    for _ in range(max_tries):
        #randomly pick the starting corner of the patch
        start_x = np.random.randint(0, orig_x - output_x + 1)
        start_y = np.random.randint(0, orig_y - output_y + 1)
        start_z = np.random.randint(0, orig_z - output_z + 1)

        #extract the label patch
        y_patch = label[start_x: start_x + output_x,
                        start_y: start_y + output_y,
                        start_z: start_z + output_z]

        # one hot encode, changing shape to (128, 128, 16, 4)
        y_onehot = tf.keras.utils.to_categorical(y_patch, num_classes = 4)

        #fraction of background voxels
        bg_ratio = np.sum(y_onehot[...,0]) / (output_x * output_y * output_z)

        if bg_ratio < background_threshold:
            #enough tumor - extract the image patch
            X_patch = np.copy(image[start_x: start_x + output_x,
                                    start_y:start_y + output_y,
                                    start_z: start_z + output_z, :])

            #move channel axis to front, (4, 128, 128, 16)
            X_patch = np.transpose(X_patch, (3, 0, 1, 2))

            #move class axis and drop the background class
            y_onehot = np.transpose(y_onehot, (3, 0, 1, 2))
            y_onehot = y_onehot[1:, :, :, :]  #dropping class 0

            return X_patch, y_onehot

    print(f"couldn't find a tumor containing patch after {max_tries}")

#sanity check
X_test, y_test = get_sub_volume(sample_image, sample_label)
print("X shape:", X_test.shape) # (4, 128, 128, 16)
print("y shape:", y_test.shape) # (3, 128, 128, 16)
print("Tumor voxels in patch:", y_test.sum())


In [ ]:
# standardize ecah channel, z-slice
def standardize(image):
    out = np.zeros_like(image)

    for c in range(image.shape[0]): #iterate along the channels
        for z in range(image.shape[3]): #iterate z-slices
            slc = image[c, :, :, z]
            mu = np.mean(slc)
            std = np.std(slc)
            if std > 0:
                out[c, :, :, z] = (slc - mu) / std
            #if std == 0, the slice is all black
    return out

#verifying that std should be ~1.0 for non zero slices
X_norm = standardize(X_test)
print("Before — mean:", X_test[0,:,:,5].mean().round(2),
" std:", X_test[0,:,:,5].std().round(2))
print("After — mean:", X_norm[0,:,:,5].mean().round(2),
" std:", X_norm[0,:,:,5].std().round(2))

In [ ]:
# data generator
# for drawing batch from randomly sampled patches

class BraTSDataGenerator(keras.utils.Sequence):

    def __init__(self, case_paths, batch_size = 2,
                 patch_x = 128, patch_y = 128, patch_z = 16, shuffle = True):
        self.case_paths = case_paths
        self.batch_size = batch_size
        self.patch_x = patch_x
        self.patch_y = patch_y
        self.patch_z = patch_z
        self.shuffle = shuffle
        self.on_epoch_end()

    def __len__(self):
        #number of batches per epoch
        return len(self.case_paths) // self.batch_size

    def on_epoch_end(self):
        #shuffle the case order after each epoch
        self.indices = np.arange(len(self.case_paths))
        if self.shuffle:
            np.random.shuffle(self.indices)

    def __getitem__(self, idx):
        #picl the batch of case indices
        batch_indices = self.indices[idx*self.batch_size : (idx +1)*self.batch_size]

        X_batch, y_batch = [], []

        for i in batch_indices:
            image, label = load_case(self.case_paths[i])
            X, y = get_sub_volume(image, label, 
                                  output_x = self.patch_x,
                                  output_y = self.patch_y,
                                  output_z = self.patch_z)
            if X is not None:
                X = standardize(X)
                X_batch.append(X)
                y_batch.append(y)

        return np.array(X_batch), np.array(y_batch)


#create generators
train_gen = BraTSDataGenerator(train_cases, batch_size = 2)
val_gen = BraTSDataGenerator(val_cases, batch_size = 2, shuffle = False)

print(f"Training batches per epoch: {len(train_gen)}")
print(f"Validation batches per epoch: {len(val_gen)}")

#Test one batch
Xb, yb = train_gen[0]
print(f"Batch X shape: {Xb.shape}") # (2, 4, 128, 128, 16)
print(f"Batch y shape: {yb.shape}") # (2, 3, 128, 128, 16)

## 3D U-Net Model

In [ ]:
#building the model
def conv_block(x, filters):
    # two consecutive 3d convolutions and ReLU activation per block
    x = Conv3D(filters, kernel_size = 3,data_format="channels_first", padding = "same")(x)
    x = Activation("relu")(x)
    x = Conv3D(filters, kernel_size = 3, data_format="channels_first", padding = "same")(x)
    x = Activation("relu")(x)
    return x

def unet_3d(input_shape = (4, 128, 128, 16), num_classes = 3, base_filters = 16):
    #building the 3d unet model

    inputs = Input(shape = input_shape)

    #encoder : each level  = 2 convs + save skip + pool

    e1 = conv_block(inputs, base_filters)
    p1 = MaxPooling3D(pool_size = 2, data_format = "channels_first")(e1)

    e2 = conv_block(p1, base_filters*2)
    p2 = MaxPooling3D(pool_size = 2, data_format = "channels_first")(e2)

    e3 = conv_block(p2, base_filters*4)
    p3 = MaxPooling3D(pool_size = 2, data_format = "channels_first")(e3)

    #bottleneck
    b = conv_block(p3, base_filters*8)

    #decoder: each level = upsample + concat skip + 2 convs

    u3 = UpSampling3D(size = 2, data_format = "channels_first")(b)
    u3 = Concatenate(axis =1)([u3, e3])
    d3 = conv_block(u3, base_filters*4)

    u2 = UpSampling3D(size = 2, data_format = "channels_first")(d3)
    u2 = Concatenate(axis = 1)([u2, e2])
    d2 = conv_block(u2, base_filters*2)

    u1 = UpSampling3D(size = 2, data_format = "channels_first")(d2)
    u1 = Concatenate(axis = 1)([u1, e1])
    d1 = conv_block(u1, base_filters)

    #output 1x1x1 conv maps to num_classes channels, sigmoid for multi-label
    outputs = Conv3D(num_classes, kernel_size = 1, padding = "same", data_format = "channels_first")(d1)
    outputs = Activation("sigmoid")(outputs)

    return Model(inputs, outputs)

model = unet_3d()
model.summary(line_length = 90)

## Loss Function and Metrics

In [ ]:
# dice coefficient
def dice_coefficient(y_true, y_pred, axis= (1, 2, 3), epsilon = 1e-5):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    #mean DC across all 3 tumor classes, higher is better
    numerator = 2 * K.sum(y_true * y_pred, axis = axis) + epsilon
    denominator = K.sum(y_true, axis = axis) + K.sum(y_pred, axis = axis) + epsilon
    #mean over the 3 classes
    return K.mean(numerator/denominator)

#soft dice loss, works with probalities for training, lower is better
def soft_dice_loss(y_true, y_pred, axis = (1, 2, 3), epsilon = 1e-5):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    numerator = 2 * K.sum(y_true * y_pred, axis = axis) + epsilon
    denominator = (K.sum(K.square(y_true), axis) + 
                   K.sum(K.square(y_pred), axis = axis) + epsilon)
    return 1 - K.mean(numerator / denominator)

a = tf.constant([[1., 0.], [0., 1.]])
b = tf.constant([[1., 1.], [0., 0.]])
print("Test Dice (expect ~0.6):", dice_coefficient(b, a, axis=(0,1)).numpy())

## Training the Model

In [ ]:
# compiling the model and defining callbacks

model.compile(
    optimizer = Adam(learning_rate = 1e-4),
    loss = soft_dice_loss,
    metrics = [dice_coefficient]
)

callbacks = [
    #save the best model weights based on validation dice
    keras.callbacks.ModelCheckpoint(
        "best_model.h5",
        monitor = "val_dice_coefficient",
        save_best_only = True,
        mode = "max",
        verbose = 1
    ),

    #reduce learning rate if val loss stops improving
    keras.callbacks.ReduceLROnPlateau(
        monitor = "val_loss",
        factor = 0.5,       #halve the learning rate
        patience = 3,       #wait 3 epochs before reducing
        min_lr = 1e-6,
        verbose = 1
    ),

    #stop early if no improvement after 8 epochs
    keras.callbacks.EarlyStopping(
        monitor = "val_loss",
        patience = 8,
        restore_best_weights = True,
        verbose = 1
    ),

    #logging the metrics to a csv 
    keras.callbacks.CSVLogger("training_log.csv")
]

print("Model compiled")
print(f"Trainable parameters: {model.count_params():,}")

In [ ]:
#run training

EPOCHS = 1

history = model.fit(
    train_gen,
    epochs = EPOCHS,
    validation_data = val_gen,
    callbacks = callbacks,
    verbose = 1
)

print("Training Complete.")
print(f"Best validation Dice: {max(history.history['val_dice_coefficient']):.4f}")

In [ ]:
#plotting training curves, loss and dice curves

import pandas as pd
log_df = pd.read_csv("training_log.csv")

fig, axes = plt.subplots(1, 2, figsize = (14,5))

#loss
axes[0].plot(log_df["loss"], label="Train Loss", color="steelblue")
axes[0].plot(log_df["val_loss"], label="Val Loss", color="tomato",
linestyle="--")
axes[0].set_title("Soft Dice Loss", fontsize=14)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(alpha=0.3)
# Dice coefficient
axes[1].plot(log_df["dice_coefficient"], label="Train Dice", color="steelblue")
axes[1].plot(log_df["val_dice_coefficient"], label="Val Dice", color="tomato",
linestyle="--")
axes[1].set_title("Dice Coefficient", fontsize=14)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Dice Score")
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.suptitle("Training History", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Final epoch stats:")
print(log_df.tail(1).to_string(index=False))